# Lenormand B15.1 — Standalone Latent Readout Confirmation

B15 的预注册候选 `25% latent + 75% B4` 只提升 `+0.00532`，没有通过 `+0.008` gate；但同一张预先输出的诊断表显示，**standalone latent probe** 达到：

- Risk Weighted-F1：`0.82444 → 0.83845`（`+0.01401`）
- Risk Macro-F1：`0.81994 → 0.82755`（`+0.00761`）
- Accuracy：`0.82299 → 0.83759`
- Behavior recall：`+0.01493`；Attempt recall：`−0.02439`

这不是把 Fold 0 偷改成最终验证：现在明确把 Fold 0 定义成 **development/screening fold**，冻结 `layer=63, C=0.01, alpha=1.0`。本 notebook 只以从未用于该选择的 Fold 1/2 判断能否进入最终系统。

确认门槛：Fold 1/2 pooled `Δ weighted-F1 ≥ +0.006`、`Δ macro-F1 ≥ 0`、两折均不得低于 `−0.003`，且 bootstrap `P(Δ>0) ≥ 0.90`。只抽取第 64 层，hidden cache 比 B15 小四倍；A100 80GB 两折预计共约 50–75 分钟。


In [ ]:
#@title 0A. 新 runtime 安装基础依赖
%%capture
!pip install -q -U   "transformers>=5.8.0"   "accelerate>=1.6.0"   "peft>=0.17.0"   "bitsandbytes>=0.46.0"   "sentencepiece>=0.2.0"   "openpyxl>=3.1.0"   "scikit-learn>=1.6.0,<1.8.0"   "scipy>=1.13.0"   "kernels"


In [ ]:
#@title 0B. Qwen3.8 kernels（安装后重启 session）
!pip install -U "flash-linear-attention[cuda]"
!pip install -U causal-conv1d --no-build-isolation
print('Runtime → Restart session；重启后从第 1 格开始。')


In [ ]:
#@title 1. Drive、更新模块与开关
from google.colab import drive, files
drive.mount('/content/drive')

from pathlib import Path
import gc, importlib, json, shutil, subprocess, sys, time

ROOT = Path('/content/drive/MyDrive/IEEE_BigData2026')
TRAIN_PATH = ROOT / 'train.xlsx'
if not TRAIN_PATH.exists(): TRAIN_PATH = ROOT / 'ieee/train.xlsx'
FOLD_SOURCE = ROOT / 'results/B4P_AVC_FAST3/B4P_CORE_OOF.npz'
B15_ROOT = ROOT / 'results/B15_LATENT_RISK_READOUT'
OUT = ROOT / 'results/B151_LATENT_STANDALONE_CONFIRMATION'
OUT.mkdir(parents=True, exist_ok=True)

FOLDS_TO_RUN = [1, 2]       # 断线后原样重跑；hidden chunks 会自动恢复

MODULE_MARKERS = {
    'b1_experiments.py': None,
    'b4p_anchor_verifier.py': 'B4P_RUNTIME_REVISION = "2026-08-21.qwen38-full64-kernels-v4"',
    'b4_task1_q38.py': 'TASK1_RUNTIME_REVISION = "2026-08-24.q38-full64-official-evidence-v4"',
    'b8_risk_only.py': 'B8R_RUNTIME_REVISION',
    # 这次版本会保存 fitted probe，供通过后直接做 test inference。
    'b15_latent_readout.py': 'joblib.dump(probe, output_dir / "B15_PROBE.joblib")',
}
stale = []
for name, marker in MODULE_MARKERS.items():
    path = ROOT / name
    if not path.exists() or (marker and marker not in path.read_text(encoding='utf-8')):
        stale.append(name)
if stale:
    print('上传并覆盖：', stale)
    uploaded = files.upload()
    for name in stale:
        if name not in uploaded: raise FileNotFoundError(name)
        shutil.copy2('/content/' + name, ROOT / name)

required = [TRAIN_PATH, FOLD_SOURCE, B15_ROOT / 'fold_0/B15_FOLD_DECISION.json']
missing = [str(path) for path in required if not path.exists()]
if missing: raise FileNotFoundError('缺少产物：\n' + '\n'.join(missing))
sys.path.insert(0, str(ROOT))
print(subprocess.run(
    ['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],
    capture_output=True, text=True,
).stdout)
print({'OUT': str(OUT), 'drive_free_gb': round(shutil.disk_usage(ROOT).free / 2**30, 2)})


In [ ]:
#@title 2. 环境、数据与 Fold-0 discovery 审计
import numpy as np
import pandas as pd
import torch
import sklearn
import transformers

import b1_experiments as b1
import b4p_anchor_verifier as b4
import b4_task1_q38 as task1
import b8_risk_only as b8
import b15_latent_readout as b15
importlib.reload(b1); importlib.reload(b4); importlib.reload(task1); importlib.reload(b8); importlib.reload(b15)

gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
kernel = b4.qwen35_kernel_status()
print({'transformers': transformers.__version__, 'torch': torch.__version__,
       'sklearn': sklearn.__version__, 'gpu_gb': gpu_gb, 'kernel': kernel})
assert gpu_gb >= 70
assert kernel['causal_conv1d'] and kernel['flash_linear_attention'], (
    '先运行 0B 并重启 session，再从第 1 格开始。'
)
torch.set_float32_matmul_precision('high')

bundle = b1.load_training_data(ROOT, TRAIN_PATH)
saved = np.load(FOLD_SOURCE, allow_pickle=True)
assert saved['row_ids'].astype(str).tolist() == bundle.row_ids.astype(str).tolist()
folds = saved['folds'].astype(int)

fold0 = json.loads((B15_ROOT / 'fold_0/B15_FOLD_DECISION.json').read_text(encoding='utf-8'))
base0 = fold0['metrics']['B4_FIXED_BLEND']
latent0 = fold0['metrics']['B15_LATENT_PROBE']
discovery = {
    'layer': int(fold0['chosen_layer']),
    'c': float(fold0['chosen_c']),
    'delta_weighted_f1': latent0['weighted_f1'] - base0['weighted_f1'],
    'delta_macro_f1': latent0['macro_f1'] - base0['macro_f1'],
    'delta_behavior_recall': latent0['per_class']['Behavior']['recall'] - base0['per_class']['Behavior']['recall'],
    'delta_attempt_recall': latent0['per_class']['Attempt']['recall'] - base0['per_class']['Attempt']['recall'],
}
print('Fold-0 development result:', discovery)
assert discovery['layer'] == 63 and abs(discovery['c'] - 0.01) < 1e-12
assert discovery['delta_weighted_f1'] >= 0.008
print('Frozen confirmation candidate: layer=63, C=0.01, alpha=1.0 (no B4 dilution)')


In [ ]:
#@title 3. 跑 / 恢复真正未见的 Fold 1、2
fold_decisions = {}
for fold in FOLDS_TO_RUN:
    cfg = b15.LatentReadoutConfig(
        fold=int(fold),
        selected_layers=(63,),       # 只保存最终层，缓存缩小 4 倍
        fixed_layer=63,
        fixed_c=0.01,
        primary_blend_alpha=1.0,     # standalone latent probe
        diagnostic_blend_alphas=(),
        extraction_batch_size=2,
        extraction_chunk_size=128,
        gate_weighted_f1_delta=0.006,
        gate_macro_f1_delta=-0.003,
        gate_behavior_recall_delta=-0.04,
        gate_attempt_recall_delta=-0.04,
    )
    started = time.perf_counter()
    result = b15.run_latent_readout_fold(
        bundle=bundle,
        folds=folds,
        root=ROOT,
        output_dir=OUT / f'fold_{fold}',
        config=cfg,
    )
    fold_decisions[int(fold)] = result
    print('fold', fold, 'elapsed min', round((time.perf_counter() - started) / 60, 1))
    display(pd.read_csv(OUT / f'fold_{fold}/B15_FOLD_SUMMARY.csv'))
    print({'fold': fold, 'deltas': result['deltas'], 'bootstrap': result['paired_bootstrap']})


In [ ]:
#@title 4. Fold 1/2 pooled confirmation（唯一最终判断）
from sklearn.metrics import f1_score

all_gold, all_base, all_latent = [], [], []
per_fold = []
for fold in (1, 2):
    path = OUT / f'fold_{fold}/B15_FOLD_OUTPUTS.npz'
    if not path.exists(): raise FileNotFoundError(f'Fold {fold} 尚未完成：{path}')
    data = np.load(path, allow_pickle=True)
    gold = data['gold'].astype(int)
    base = data['prediction__B4_FIXED_BLEND'].astype(int)
    latent = data['prediction__B15_LATENT_BLEND_100'].astype(int)
    base_m = b8.risk_metrics(gold, base)
    latent_m = b8.risk_metrics(gold, latent)
    per_fold.append({
        'fold': fold,
        'base_weighted_f1': base_m['weighted_f1'],
        'latent_weighted_f1': latent_m['weighted_f1'],
        'delta_weighted_f1': latent_m['weighted_f1'] - base_m['weighted_f1'],
        'base_macro_f1': base_m['macro_f1'],
        'latent_macro_f1': latent_m['macro_f1'],
        'delta_macro_f1': latent_m['macro_f1'] - base_m['macro_f1'],
    })
    all_gold.append(gold); all_base.append(base); all_latent.append(latent)

gold = np.concatenate(all_gold)
base = np.concatenate(all_base)
latent = np.concatenate(all_latent)
base_metrics = b8.risk_metrics(gold, base)
latent_metrics = b8.risk_metrics(gold, latent)
bootstrap = b8._paired_bootstrap_delta(gold, base, latent, 3000, 151)
fold_frame = pd.DataFrame(per_fold)
display(fold_frame)

delta_w = latent_metrics['weighted_f1'] - base_metrics['weighted_f1']
delta_m = latent_metrics['macro_f1'] - base_metrics['macro_f1']
passed = bool(
    delta_w >= 0.006
    and delta_m >= 0.0
    and fold_frame.delta_weighted_f1.min() >= -0.003
    and bootstrap['probability_positive'] >= 0.90
)
decision = {
    'runtime_revision': '2026-08-30.b151-standalone-confirmation-v1',
    'development_fold': 0,
    'confirmation_folds': [1, 2],
    'frozen_candidate': {'layer': 63, 'c': 0.01, 'alpha': 1.0},
    'baseline': base_metrics,
    'latent': latent_metrics,
    'delta_weighted_f1': delta_w,
    'delta_macro_f1': delta_m,
    'bootstrap': bootstrap,
    'per_fold': per_fold,
    'passed': passed,
    'decision': 'BUILD_B151_TEST_ENSEMBLE' if passed else 'STOP_B151',
    'acceptance_rule': {
        'pooled_delta_weighted_f1_min': 0.006,
        'pooled_delta_macro_f1_min': 0.0,
        'each_fold_delta_weighted_f1_min': -0.003,
        'bootstrap_probability_positive_min': 0.90,
    },
}
fold_frame.to_csv(OUT / 'B151_CONFIRMATION_FOLDS.csv', index=False)
(OUT / 'B151_CONFIRMATION_DECISION.json').write_text(
    json.dumps(decision, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(json.dumps(decision, indent=2))
print('PASS：下一步只做 test hidden extraction + 三折 probe ensemble。' if passed else 'FAIL：停止 latent 路线。')


In [ ]:
#@title 5. 下载轻量结果包（不包含 hidden chunks）
import shutil

package = Path('/content/B151_LATENT_CONFIRMATION_REPORT')
if package.exists(): shutil.rmtree(package)
package.mkdir(parents=True)
for fold in (1, 2):
    source = OUT / f'fold_{fold}'
    target = package / f'fold_{fold}'
    target.mkdir()
    for name in (
        'B15_FOLD_DECISION.json', 'B15_FOLD_SUMMARY.csv',
        'B15_FOLD_PREDICTIONS.csv', 'B15_FOLD_OUTPUTS.npz',
        'B15_PROBE.joblib', 'B15_PROMPT_AUDIT.csv',
    ):
        if (source / name).exists(): shutil.copy2(source / name, target / name)
for name in ('B151_CONFIRMATION_FOLDS.csv', 'B151_CONFIRMATION_DECISION.json'):
    if (OUT / name).exists(): shutil.copy2(OUT / name, package / name)
archive = shutil.make_archive('/content/B151_LATENT_CONFIRMATION_REPORT', 'zip', package)
print(archive)
files.download(archive)
